# Example 09: PyTorch-Lightning

## 事前準備

In [1]:
import torch

# GPUが使えるか確認してデバイスを設定
# NOTE: `x = x.to(device) ` とすることで対象のデバイスに切り替え可能
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from typing import Callable

import os
import numpy as np
import random
import torch
import pytorch_lightning as pl


# シード値の固定
# NOTE: 戻り値はDataLoaderのシード固定に使用する。　
# 　　　　　　　　　　　　(ex) loader = DataLoader(..., worker_init_fn=seed_worker, generator=generator)
#       また、plのTrainer(deterministic=True)とする必要がある。
def fixing_seed(seed: int=42) -> tuple[torch.Generator, Callable]:
    # Python のシード固定
    random.seed(seed)
    # Numpy のシード固定
    np.random.seed(seed)
    # PyTorch のシード固定
    torch.manual_seed(seed)
    # CUDA の再現性確保の設定
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # PyTorch Lightning のシード固定
    pl.seed_everything(seed, workers=True)

In [3]:
SEED = 24

fixing_seed(SEED)

Seed set to 24


In [4]:
NUM_WORKERS = 9

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")

## CNN

In [5]:
from typing import Any

import numpy as np
import optuna
import torch
import torchinfo
import pytorch_lightning as pl
from torch import nn
from torch import optim
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, SubsetRandomSampler
from lightning.pytorch.loggers import MLFlowLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from datetime import datetime
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

### モデル構築

In [6]:
class Model(nn.Module):
    def __init__(self, n_classes: int,
                       dropout_prob: float = 0.5,
                       activation_func: nn.Module = nn.ReLU()):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 5)         # 入力チャネル、出力チャネル、フィルタ数
        self.active = activation_func
        self.pool = nn.MaxPool2d(2, 2)          # 領域のサイズ、領域の間隔
        self.conv2 = nn.Conv2d(8, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 256)
        self.dropout = nn.Dropout(dropout_prob)          # ドロップアウト率
        self.fc2 = nn.Linear(256, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.active(self.conv1(x))
        x = self.pool(x)
        x = self.active(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 16 * 5 * 5)
        x = self.active(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### PyTorch Lightning モデル用意

#### データモジュール

In [7]:
class CIFAR10DataModule(pl.LightningDataModule):
    def __init__(self, root_dir: str="../cache/data", batch_size: int=128, valid_size: float=0.2, num_workers: int=0, seed=24):
        super().__init__()
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.valid_size = valid_size
        self.num_workers = num_workers
        
        self.train_transform = transforms.Compose([
            transforms.RandomAffine((-30, 30), scale=(0.8, 1.2)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
        ])
        self.valid_transform = transforms.Compose([
            transforms.RandomAffine((-30, 30), scale=(0.8, 1.2)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
        ])
        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
        ])

        # DataLoader の乱数源
        self.torch_generator = torch.Generator()
        self.torch_generator.manual_seed(seed)

    def _worker_init_fn(self, worker_id: int) -> None:
        # 各 worker で Python / NumPy も固定
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    def _prepare_train_valid_index(self, data_length: int, valid_size: float = 0.2) -> tuple[SubsetRandomSampler, SubsetRandomSampler]:
        # 学習データをtrainとvalidに分割するためのインデックスを準備
        indices = list(range(data_length))
        split = int(np.floor(valid_size * data_length))
        np.random.shuffle(indices)

        train_idx = indices[split:]
        valid_idx = indices[:split]

        train_sampler = SubsetRandomSampler(train_idx)
        validation_sampler = SubsetRandomSampler(valid_idx)
        return (train_sampler, validation_sampler)

    def prepare_data(self) -> None:
        train_dataset = CIFAR10(self.root_dir, train=True, download=True)
        CIFAR10(self.root_dir, train=False, download=True)
        self.train_sampler, self.valid_sampler = self._prepare_train_valid_index(len(train_dataset), self.valid_size)

    def setup(self, stage: str) -> None:
        if stage == "fit":
            self.train_dataset = CIFAR10(self.root_dir, train=True, transform=self.train_transform)
            self.valid_dataset = CIFAR10(self.root_dir, train=True, transform=self.valid_transform)
        elif stage == "test":
            self.test_dataset = CIFAR10(self.root_dir, train=False, transform=self.test_transform)

    def train_dataloader(self) -> DataLoader:
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, sampler=self.train_sampler,
                          generator=self.torch_generator, worker_init_fn=self._worker_init_fn)

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.valid_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, sampler=self.valid_sampler,
                          generator=self.torch_generator, worker_init_fn=self._worker_init_fn)

    def test_dataloader(self) -> DataLoader:
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers,
                          generator=self.torch_generator, worker_init_fn=self._worker_init_fn)

In [8]:
def check_datamodule():
    dm = CIFAR10DataModule()
    dm.prepare_data()
    for stage in ("fit", "test"):
        dm.setup(stage)
        print(f"///// {stage} /////")
        print(dm)
        print()

check_datamodule()

///// fit /////
{Train dataloader: size=50000}
{Validation dataloader: size=50000}
{Test dataloader: None}
{Predict dataloader: None}

///// test /////
{Train dataloader: size=50000}
{Validation dataloader: size=50000}
{Test dataloader: size=10000}
{Predict dataloader: None}



#### ネットワークモジュール

In [9]:
def calc_accuracy(output: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(target.view_as(pred)).sum().item()
    accuracy = correct / output.size(0)
    return accuracy


class LitModel(pl.LightningModule):

    def __init__(self, n_class: int = 10, batch_size: int = 128):
        super().__init__()
        self.batch_size = batch_size
        self.model = Model(n_class)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, data: torch.Tensor) -> torch.Tensor:
        return self.model(data)

    def training_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        data, target = batch
        output = self.forward(data)
        loss = self.criterion(output, target)
        accuracy = calc_accuracy(output, target)
        self.log('train_loss', loss)
        self.log('train_accuracy', accuracy)
        return loss

    def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        data, target = batch
        output = self.forward(data)
        loss = self.criterion(output, target)
        accuracy = calc_accuracy(output, target)
        self.log('valid_loss', loss)
        self.log('valid_accuracy', accuracy)
        return loss

    def test_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        data, target = batch
        output = self.forward(data)
        loss = self.criterion(output, target)
        accuracy = calc_accuracy(output, target)
        self.log('test_loss', loss)
        self.log('test_accuracy', accuracy)
        return loss

    def configure_optimizers(self):
        # Generate the optimizers.
        return optim.Adam(self.model.parameters(), lr=0.001)

In [10]:
!ls /mlruns/artifacts/example009_pytorch-lightning/checkpoints

ls: cannot access '/mlruns/artifacts/example009_pytorch-lightning/checkpoints': No such file or directory


In [11]:
dm = CIFAR10DataModule(num_workers=NUM_WORKERS, seed=SEED)
dm.prepare_data()
dm.setup(stage="fit")

model = LitModel()

run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

mlflow_logger = MLFlowLogger(
    experiment_name="example009_pytorch-lightning",
    run_name=run_name,
    tracking_uri=MLFLOW_TRACKING_URI
)
checkpoint_callback = ModelCheckpoint(
    dirpath="/mlruns/artifacts/example009_pytorch-lightning/checkpoints/",
    filename="epoch={epoch}-step={step}",
    save_top_k=-1,           # 全て保存
    every_n_epochs=1,        # 毎エポック保存
)

trainer = pl.Trainer(
    max_epochs=30,
    deterministic=True,
    logger=mlflow_logger,
    callbacks=[checkpoint_callback],
)

trainer.fit(model, dm)
trainer.validate(datamodule=dm)

dm.setup(stage="test")
trainer.test(datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | Model            | 109 K  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
109 K     Trainable params
0         Non-trainable params
109 K     Total params
0.436     Total estimated model params size (MB)
9         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

/opt/venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/checkpoint_connector.py:149: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at /mlruns/artifacts/example009_pytorch-lightning/checkpoints/epoch=epoch=29-step=step=9390.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /mlruns/artifacts/example009_pytorch-lightning/checkpoints/epoch=epoch=29-step=step=9390.ckpt
/opt/venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:484: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.5936999917030334     │
│         test_loss         │    1.1342898607254028     │
└───────────────────────────┴───────────────────────────┘

🏃 View run 20251117_231448 at: http://mlflow:5000/#/experiments/5/runs/1445cd0c1fa947c29776965924581e09
🧪 View experiment at: http://mlflow:5000/#/experiments/5


[{'test_loss': 1.1342898607254028, 'test_accuracy': 0.5936999917030334}]